<a href="https://colab.research.google.com/github/bkkaggle/pytorch-CycleGAN-and-pix2pix/blob/master/CycleGAN.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Setup

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
import os
os.chdir('/content/drive/MyDrive/lora-film-multitask-cyclegan/') # change to wherever you cloned yours

# Obtain Data

## Download Data to your Drive

Run this .sh file to download the .zip files containing the datasets. The .zip files can be used directly without unzipping.

Alternatively you can download the files manually from https://drive.google.com/drive/folders/1rs6s4e_eyTJCuE9bMdAJuAh1Isis2PfA?usp=sharing. These files must exists in a folder called datasets, as such: lora-film-multitask-cyclegan/datasets

In [ ]:
!bash download_datasets.sh

## Make Data available on Local Colab Session

In [ ]:
# Semantically Unrelated Datasets
#!unzip -oq "/content/drive/MyDrive/lora-film-multitask-cyclegan/datasets/horse2zebra.zip" -d /content/data
#!unzip -oq "/content/drive/MyDrive/lora-film-multitask-cyclegan/datasets/day2night.zip" -d /content/data
#!unzip -oq "/content/drive/MyDrive/lora-film-multitask-cyclegan/datasets/summer2winter_yosemite.zip" -d /content/data
#!unzip -oq "/content/drive/MyDrive/lora-film-multitask-cyclegan/datasets/cat2dog_reduced.zip" -d /content/data

# Semantically Related Dataset
!unzip -oq "/content/drive/MyDrive/lora-film-multitask-cyclegan/datasets/cat2dog_reduced.zip" -d /content/data # cat2dog is the same in both
!unzip -oq "/content/drive/MyDrive/lora-film-multitask-cyclegan/datasets/fox2wolf.zip" -d /content/data
!unzip -oq "/content/drive/MyDrive/lora-film-multitask-cyclegan/datasets/tiger2lion.zip" -d /content/data
!unzip -oq "/content/drive/MyDrive/lora-film-multitask-cyclegan/datasets/cheetah2leopard.zip" -d /content/data
!unzip -oq "/content/drive/MyDrive/lora-film-multitask-cyclegan/datasets/wolf2dog.zip" -d /content/data

!ls /content/data/

In [ ]:
for d in ["cat2dog_reduced", "cheetah2leopard", "fox2wolf", "tiger2lion", "wolf2dog", "horse2zebra", "day2night", "summer2winter_yosemite", "apple2orange"]:
    print(f"\n📁 {d}")
    for s in ["trainA", "testB","trainB", "testA"]:
          p = f"/content/data/{d}/{s}"
          if os.path.isdir(p):
              print(f"  {s}: {len(os.listdir(p))}")


# Training

Install requirements

In [ ]:
!pip install torch torchvision
!pip install dominate visdom
!pip install torch_fidelity

## Train STT

In [ ]:
from datetime import datetime
import os

n_epochs = 100
n_epochs_decay = 101
task = 'tiger2lion'
model_name = "STT_" + task + '_' + datetime.now().strftime("%d%m%Y_%H%M")
print(model_name)

num_threads = os.cpu_count()
print('cpu cores:', num_threads)

In [ ]:
!python models/STT/train.py \
      --dataroot /content/data/{task} \
      --name {model_name} \
      --netG resnet_9blocks \
      --netD basic \
      --n_epochs {n_epochs} \
      --n_epochs_decay {n_epochs_decay} \
      --num_threads {num_threads}

## Train FiLM-MTT

In [ ]:
from datetime import datetime
import os

n_epochs = 100
n_epochs_decay = 101
tasks = ['fox2wolf', 'tiger2lion', 'cheetah2leopard', 'cat2dog_reduced', 'wolf2dog']
tasks_str = ' '.join(tasks)
model_name = "FiLM_MTT_" + '_'.join(tasks) + '_' + datetime.now().strftime("%d%m%Y_%H%M")
print(model_name)

num_threads = os.cpu_count()
print('cpu cores:', num_threads)

In [ ]:
!python models/FiLM-MTT/train-per-batch.py \
  --dataroot_general /content/data \
  --tasks {tasks_str} \
  --name {model_name} \
  --netG resnet_9blocks_film \
  --netD basic_film \
  --n_epochs {n_epochs} \
  --n_epochs_decay {n_epochs_decay} \
  --fid_freq 10 \
  --num_threads {num_threads}

## Train LoRA-MTT

In [ ]:
from datetime import datetime
import os

n_epochs = 5
n_epochs_decay = 5
tasks = ['fox2wolf', 'tiger2lion', 'cheetah2leopard', 'cat2dog_reduced', 'wolf2dog']
tasks_str = ' '.join(tasks)
model_name = "LoRA_MTT_" + '_'.join(tasks) + '_' + datetime.now().strftime("%d%m%Y_%H%M")
print(model_name)

num_threads = os.cpu_count()
print('cpu cores:', num_threads)

In [ ]:
!python models/LoRA-MTT/train-per-batch.py \
  --dataroot_general /content/data \
  --tasks {tasks_str} \
  --name {model_name} \
  --netG resnet_9blocks_lora \
  --netD basic_film \
  --n_epochs {n_epochs} \
  --n_epochs_decay {n_epochs_decay} \
  --fid_freq 10 \
  --num_threads {num_threads}

## Train FT-LoRA-MTT

### Step 1: Train Pre-trained LoRA-MTT

In [ ]:
from datetime import datetime
import os

n_epochs = 100
n_epochs_decay = 101
tasks = ['fox2wolf', 'tiger2lion', 'cheetah2leopard', 'cat2dog_reduced', 'wolf2dog']
tasks_str = ' '.join(tasks)
model_name = "Pretrained_FT_LoRA_MTT_" + '_'.join(tasks) + '_' + datetime.now().strftime("%d%m%Y_%H%M")
print(model_name)

num_threads = os.cpu_count()
print('cpu cores:', num_threads)

In [ ]:
!python models/LoRA-MTT/train-per-batch.py \
  --dataroot_general /content/data \
  --tasks {tasks_str} \
  --name {model_name} \
  --netG resnet_9blocks_lora \
  --netD basic_film \
  --n_epochs {n_epochs} \
  --n_epochs_decay {n_epochs_decay} \
  --fid_freq 10 \
  --num_threads {num_threads}

### Step 2: Fine-tune Pre-trained LoRA-MTT

In [ ]:
from datetime import datetime
import os

n_epochs = 5
n_epochs_decay = 5
finetune_task = 'cat2dog_reduced'
pretrain_name = "Pretrained_FT_LoRA_MTT_fox2wolf_tiger2lion_cheetah2leopard_cat2dog_reduced_wolf2dog14042026_1110" # Change this with your pretrained model
model_name = "Finetune_FT_LoRA_MTT_" + datetime.now().strftime("%d%m%Y_%H%M")
print(model_name)

num_threads = os.cpu_count()
print('cpu cores:', num_threads)

In [ ]:
!python models/FT-LoRA-MTT/train-per-batch.py \
  --dataroot_general /content/data \
  --use_lora \
  --pretrained_name {pretrain_name} \
  --finetune_lora {finetune_task} \
  --name {model_name} \
  --netG resnet_9blocks_lora \
  --netD basic_film \
  --n_epochs {n_epochs} \
  --n_epochs_decay {n_epochs_decay} \
  --fid_freq 10 \
  --num_threads {num_threads}

## Stop run time

In [ ]:
from google.colab import drive
from google.colab import runtime
drive.flush_and_unmount()
runtime.unassign()